# Migration Phase 5: end-to-end `DataPreparationPipeline`

`config.py` and `pipeline.py` are rewritten around the new sources:

- **`PipelineConfig`**: `paths.brainlink_db`, `paths.tabular_derivatives_root`,
  `atlas_name`, `anat_atlases`, `session_variant`, `labs`,
  `require_complete_mapping`, `force` - no more `sessions_csv`/`cat12_root`/
  `qsiparc_path`/`qsirecon_path`.
- **`DataPreparationPipeline.run()`**:
  1. `BehavioralLoader.get_sessions()` -> canonical session list (`uid`,
     `session_id`, `AGE`, `sex`, ...).
  2. `TabularDerivativesLoader.load_anatomical/load_diffusion()` -> long
     format for sessions not yet in the store (incremental, via
     `get_existing_sessions()`).
  3. `FeatureStore.save_anatomical_long/save_diffusion_long/save_tiv/save_metadata`.
  4. `FeatureStore.generate_wide_features()`.

This notebook runs `run()` on a small, lab-restricted slice (`labs=["TS"]`,
the smallest lab) into a demo store under `data/processed/` (gitignored),
then re-runs it to demonstrate incremental skipping.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv(Path.cwd().parent / ".env")

from neuroalign.data.preprocessing import (
    PipelineConfig,
    DataPaths,
    ModalityConfig,
    DataPreparationPipeline,
)

store_dir = Path.cwd().parent / "data" / "processed" / "phase5_demo"

config = PipelineConfig(
    paths=DataPaths(
        brainlink_db=os.environ["BRAINLINK_DB_PATH"],
        tabular_derivatives_root=os.environ["TABULAR_DERIVATIVES_ROOT"],
        output_dir=store_dir,
    ),
    modalities=ModalityConfig(anatomical=True, diffusion=True),
    atlas_name=os.environ["ATLAS_NAME"],
    anat_atlases=tuple(os.environ["ANAT_ATLASES"].split(",")),
    session_variant=os.environ["SESSION_VARIANT"],
    labs=["TS"],
)
config

PipelineConfig(paths=DataPaths(brainlink_db=PosixPath('/media/storage/brainlink/brainlink.db'), tabular_derivatives_root=PosixPath('/mnt/62/Processed_Data/derivatives/tabular'), output_dir=PosixPath('/home/galkepler/Projects/neuroalign/data/processed/phase5_demo')), modalities=ModalityConfig(anatomical=True, diffusion=True), output=OutputConfig(prefix='neuroalign', compression='snappy'), atlas_name='Schaefer2018N400n7Tian2020S2', anat_atlases=('Schaefer2018N400n7', 'Tian2020S2'), session_variant='cross', labs=['TS'], require_complete_mapping=True, force=False)

## 1. First run

Loads every `lab="TS"` session (smallest lab, ~tens of sessions) and builds
the store from scratch.

In [2]:
pipeline = DataPreparationPipeline(config)
result = pipeline.run()
print(f"n_new_sessions={result.n_new_sessions}, n_skipped_sessions={result.n_skipped_sessions}")
print(f"long_formats_saved={result.long_formats_saved}")
print(f"n_wide_features={len(result.wide_features_generated)}")
result.metadata

n_new_sessions=46, n_skipped_sessions=0
long_formats_saved=['anatomical', 'diffusion_AMICONODDI', 'diffusion_DIPYDKI', 'diffusion_DIPYMAPMRI', 'diffusion_DSIStudio', 'diffusion_MRtrix3actHSVS']
n_wide_features=1278


{'n_subjects': 19,
 'n_sessions': 22,
 'long_formats': ['anatomical',
  'diffusion_AMICONODDI',
  'diffusion_DIPYDKI',
  'diffusion_DIPYMAPMRI',
  'diffusion_DSIStudio',
  'diffusion_MRtrix3actHSVS'],
 'n_wide_features': 1278,
 'anatomical_features': ['tiv',
  'anat_num_vertices',
  'anat_surface_area_mm2',
  'anat_gray_matter_volume_mm3',
  'anat_thickness_mean_mm',
  'anat_thickness_std_mm',
  'anat_mean_curvature',
  'anat_gaussian_curvature',
  'anat_folding_index',
  'anat_curvature_index',
  'anat_white_surf_area_mm2',
  'anat_brain_seg_vol_mm3',
  'anat_brain_seg_no_vent_mm3',
  'anat_cortex_vol_mm3',
  'anat_supratentorial_vol_mm3',
  'anat_num_voxels',
  'anat_volume_mm3',
  'anat_intensity_mean',
  'anat_intensity_std',
  'anat_intensity_min',
  'anat_intensity_max',
  'anat_intensity_range',
  'anat_intensity_snr',
  'anat_subcort_gray_mm3'],
 'diffusion_features': ['AMICONODDI_noddi_direction_mean',
  'AMICONODDI_noddi_direction_std',
  'AMICONODDI_noddi_direction_median',


## 2. Second run (incremental)

All sessions are already in the store, so `run()` should report
`n_new_sessions=0` and skip loading.

In [3]:
pipeline2 = DataPreparationPipeline(config)
result2 = pipeline2.run()
print(f"n_new_sessions={result2.n_new_sessions}, n_skipped_sessions={result2.n_skipped_sessions}")

n_new_sessions=0, n_skipped_sessions=46


## 3. Inspect the store

In [4]:
store = result.store
print("long formats:", store.list_long_formats())
print("anatomical features (first 5):", store.list_features("anatomical")[:5])
print("diffusion features (first 5):", store.list_features("diffusion")[:5])

ct = store.load_feature("anat_thickness_mean_mm", include_tiv=True)
ct[["uid", "session_id", "AGE", "sex", "tiv_mm3"]].head()

long formats: ['anatomical', 'diffusion_AMICONODDI', 'diffusion_DIPYDKI', 'diffusion_DIPYMAPMRI', 'diffusion_DSIStudio', 'diffusion_MRtrix3actHSVS']
anatomical features (first 5): ['anat_brain_seg_no_vent_mm3', 'anat_brain_seg_vol_mm3', 'anat_cortex_vol_mm3', 'anat_curvature_index', 'anat_folding_index']
diffusion features (first 5): ['AMICONODDI_noddi_direction_coverage', 'AMICONODDI_noddi_direction_cv', 'AMICONODDI_noddi_direction_excess_kurtosis', 'AMICONODDI_noddi_direction_iqr_filtered_mean', 'AMICONODDI_noddi_direction_iqr_filtered_std']


,uid,session_id,AGE,sex,tiv_mm3
0,S028694,202409261650,22.99,Female,1.401337e+06
1,S028694,202507241349,23.81,Female,1.373499e+06
2,S067082,202509031744,37.49,Female,1.422386e+06
3,S076379,202410131245,22.26,Female,1.500654e+06
4,S108449,202510160926,31.28,Female,1.343475e+06


## 4. Store summary

In [5]:
store.summary()

{'root_dir': '/home/galkepler/Projects/neuroalign/data/processed/phase5_demo',
 'atlas_name': 'Schaefer2018N400n7Tian2020S2',
 'n_sessions': 22,
 'n_subjects': 19,
 'long_formats': ['anatomical',
  'diffusion_AMICONODDI',
  'diffusion_DIPYDKI',
  'diffusion_DIPYMAPMRI',
  'diffusion_DSIStudio',
  'diffusion_MRtrix3actHSVS'],
 'n_wide_features': 1278,
 'anatomical_features': ['tiv',
  'anat_num_vertices',
  'anat_surface_area_mm2',
  'anat_gray_matter_volume_mm3',
  'anat_thickness_mean_mm',
  'anat_thickness_std_mm',
  'anat_mean_curvature',
  'anat_gaussian_curvature',
  'anat_folding_index',
  'anat_curvature_index',
  'anat_white_surf_area_mm2',
  'anat_brain_seg_vol_mm3',
  'anat_brain_seg_no_vent_mm3',
  'anat_cortex_vol_mm3',
  'anat_supratentorial_vol_mm3',
  'anat_num_voxels',
  'anat_volume_mm3',
  'anat_intensity_mean',
  'anat_intensity_std',
  'anat_intensity_min',
  'anat_intensity_max',
  'anat_intensity_range',
  'anat_intensity_snr',
  'anat_subcort_gray_mm3'],
 'diffus